1. Import Libraries

In [ ]:
import pandas as pd # Data manipulation
import numpy as np # Numerical operations
import re # Regular expressions

# NLP
import nltk # Natural Language Toolkit
from nltk.corpus import stopwords # Stopwords for text preprocessing
from nltk.stem import PorterStemmer # Stemming for text preprocessing

# ML
from sklearn.model_selection import train_test_split # Train-test split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer # Text vectorization
from sklearn.linear_model import LogisticRegression # Logistic Regression classifier
from sklearn.naive_bayes import MultinomialNB # Naive Bayes classifier
from sklearn.tree import DecisionTreeClassifier # Decision Tree classifier

# Metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score # Evaluation metrics

In [3]:
nltk.download('stopwords') # Download stopwords from NLTK

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jaysh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

2. Load Dataset (Kaggle)

In [5]:
df = pd.read_csv("IMDB_Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


3. Data Understanding

In [7]:
print("Shape:", df.shape) # Number of rows and columns

print("\nClass Distribution:") 
print(df['sentiment'].value_counts()) # Count of positive and negative reviews

print("\nSample Text:")
print(df['review'][0]) # Print the first review to see the raw text

Shape: (50000, 2)

Class Distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Sample Text:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares,

4. NLP Preprocessing

In [8]:
stop_words = set(stopwords.words('english')) # Create a set of English stopwords for faster lookup
stemmer = PorterStemmer() # Initialize the Porter Stemmer for stemming words

def preprocess_text(text): # Function to preprocess text by lowercasing, removing URLs, punctuation, stopwords, and applying stemming
    text = text.lower() # Convert text to lowercase
    text = re.sub(r"http\S+|www\S+", "", text) # Remove URLs using regular expressions
    text = re.sub(r"[^\w\s]", "", text) # Remove punctuation using regular expressions

    words = text.split() # Split the text into individual words

    words = [word for word in words if word not in stop_words] # Remove stopwords from the list of words

    words = [stemmer.stem(word) for word in words] # Apply stemming to each word in the list

    return " ".join(words) # Join the processed words back into a single string and return it

Apply preprocessing

In [ ]:
df['clean_text'] = df['review'].apply(preprocess_text) # Apply the preprocessing function to the 'review' column and create a new 'clean_text' column with the processed text

5. Feature Engineering

Bag of Words

In [10]:
bow = CountVectorizer(max_features=5000) # Initialize CountVectorizer with a maximum of 5000 features to convert text to a bag-of-words representation
X_bow = bow.fit_transform(df['clean_text']) # Fit the CountVectorizer to the cleaned text and transform it into a sparse matrix of token counts

TF-IDF

In [11]:
tfidf = TfidfVectorizer(max_features=5000) # Initialize TfidfVectorizer with a maximum of 5000 features to convert text to a TF-IDF representation
X_tfidf = tfidf.fit_transform(df['clean_text']) # Fit the TfidfVectorizer to the cleaned text and transform it into a sparse matrix of TF-IDF features

6. Prepare Labels

In [12]:
y = df['sentiment'].map({'positive':1, 'negative':0}) # Map sentiment labels to binary values (1 for positive, 0 for negative)

7. Train-Test Split

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
) # Split the data into training and testing sets with an 80-20 split and a fixed random state for reproducibility

8. Model Training

Logistic Regression

In [ ]:
lr = LogisticRegression() # Initialize the Logistic Regression classifier
lr.fit(X_train, y_train) # Fit the Logistic Regression model to the training data
y_pred_lr = lr.predict(X_test) # Predict the labels for the test set using the trained Logistic Regression model

Naive Bayes

In [ ]:
nb = MultinomialNB() # Initialize the Multinomial Naive Bayes classifier
nb.fit(X_train, y_train) # Fit the Multinomial Naive Bayes model to the training data
y_pred_nb = nb.predict(X_test) # Predict the labels for the test set using the trained Multinomial Naive Bayes model

Decision Tree

In [ ]:
dt = DecisionTreeClassifier() # Initialize the Decision Tree classifier
dt.fit(X_train, y_train) # Fit the Decision Tree model to the training data
y_pred_dt = dt.predict(X_test) # Predict the labels for the test set using the trained Decision Tree model

9. Evaluation Function

In [ ]:
def evaluate_model(y_test, y_pred, name):
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, y_pred)) # Print the accuracy of the model
    print("Precision:", precision_score(y_test, y_pred)) # Print the precision of the model
    print("Recall:", recall_score(y_test, y_pred)) # Print the recall of the model
    print("F1 Score:", f1_score(y_test, y_pred)) # Print the F1 score of the model

Evaluate all models

In [ ]:
evaluate_model(y_test, y_pred_lr, "Logistic Regression") # Evaluate the Logistic Regression model using the defined evaluation function
evaluate_model(y_test, y_pred_nb, "Naive Bayes") # Evaluate the Naive Bayes model using the defined evaluation function
evaluate_model(y_test, y_pred_dt, "Decision Tree") # Evaluate the Decision Tree model using the defined evaluation function


Logistic Regression
Accuracy: 0.886
Precision: 0.8752646775745909
Recall: 0.9023615796785076
F1 Score: 0.8886066054328708

Naive Bayes
Accuracy: 0.8494
Precision: 0.8456270788495402
Recall: 0.8577098630680691
F1 Score: 0.8516256157635468

Decision Tree
Accuracy: 0.7168
Precision: 0.7246996538383221
Recall: 0.7062909307402262
F1 Score: 0.7153768844221106


10. Comparison Table

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Naive Bayes', 'Decision Tree'], 
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_dt)
    ]
})

results

,Model,Accuracy
0,Logistic Regression,0.8860
1,Naive Bayes,0.8494
2,Decision Tree,0.7168


11. Insights

1. TF-IDF performed better than BoW
2. Logistic Regression gave highest accuracy
3. Naive Bayes was fastest
4. Decision Tree overfitted

12. Final Pipeline Flow

Raw Data → Preprocessing → TF-IDF → Model Training → Evaluation → Comparison